# N=43

Test flipping - check orientation

In [ ]:
import os
import nibabel as nib
import json
import numpy as np

input_folder = '/mnt/sda1/Repos/a-eye/Output/mri_qc/samples/samples_v3_bids'

# Iterate through all NIfTI files in the folder
for root, dirs, files in os.walk(input_folder):
    # Skip any folder that contains 'derivatives' in its path
    if 'derivatives' in root:
        continue
    for file in files:
        if file.endswith('.nii.gz'):
            img_path = os.path.join(root, file)
            img = nib.load(img_path)
            print(f"File: {img_path}")
            print(f"Orientation: {nib.aff2axcodes(img.affine)}")
            print(f"Shape: {img.shape}\n")

Flip axis in MR images

In [5]:
import os
import nibabel as nib
import json

input_folder = '/mnt/sda1/Repos/a-eye/Output/mri_qc/samples/samples_v3_bids'
output_folder = '/home/jaimebarranco/Desktop/tmp/test_inference_flipped_images/flipped_images_affine_corrected'

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Function to flip the axis of the image without flipping the image itself
def flip_axis(img):
    data = img.get_fdata()
    flipped_data = data[::-1, :, :]
    
    # Update affine to account for the flip
    flipped_affine = img.affine.copy()
    flipped_affine[0, 3] -= (img.shape[0] - 1) * img.affine[0, 0]
    
    flipped_img = nib.Nifti1Image(flipped_data, flipped_affine, img.header)
    return flipped_img

# Iterate through the BIDS folder to find NIfTI and JSON files
for root, dirs, files in os.walk(input_folder):
    # Skip any folder that contains 'derivatives' in its path
    if 'derivatives' in root:
        continue
    for file in files:
        if file.endswith('.nii.gz') and 'mask' not in file:
            nifti_path = os.path.join(root, file)
            json_path = nifti_path.replace('.nii.gz', '.json')
            
            # Load the NIfTI image
            img = nib.load(nifti_path)
            
            # Flip the axis
            flipped_img = flip_axis(img)
            
            # Create the same folder structure in the output folder
            relative_path = os.path.relpath(root, input_folder)
            subject_folder = os.path.join(output_folder, relative_path)
            os.makedirs(subject_folder, exist_ok=True)
            
            # Save the flipped image
            flipped_nifti_path = os.path.join(subject_folder, os.path.basename(nifti_path).replace('.nii.gz', '_flipped.nii.gz'))
            nib.save(flipped_img, flipped_nifti_path)
            
            # Load and save the JSON metadata
            if os.path.exists(json_path):
                with open(json_path, 'r') as f:
                    metadata = json.load(f)
                flipped_json_path = os.path.join(subject_folder, os.path.basename(json_path).replace('.json', '_flipped.json'))
                with open(flipped_json_path, 'w') as f:
                    json.dump(metadata, f, indent=4)

Sanity check - ITK: Identity matrix & Determinant ≈ ±1

In [6]:
import numpy as np
R = flipped_img.affine[:3, :3]
print("Orthogonality check:")
print(np.round(R.T @ R, 5))
print("Determinant:", np.linalg.det(R))

Orthogonality check:
[[ 1.  0. -0.]
 [ 0.  1. -0.]
 [-0. -0.  1.]]
Determinant: 1.0000002490881514


Crop left upper quadrant

In [ ]:
import os
import nibabel as nib

input_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/images"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/images_cropped_left_quadrant"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        img = nib.load(path)
        data = img.get_fdata()
        
        # Crop to left upper quadrant cube (RAS orientation)
        # Left: reduce right dimension (axis 0)
        # Upper: reduce inferior dimension (axis 2)
        # Anterior: reduce posterior dimension (axis 1)
        mid_x = data.shape[0] // 2
        mid_y = data.shape[1] // 2
        mid_z = data.shape[2] // 2
        cropped_data = data[:mid_x, mid_y:, :]
        
        cropped_img = nib.Nifti1Image(
            cropped_data,
            img.affine,
            img.header
        )
        
        out_path = os.path.join(output_folder, file.replace(".nii.gz", "_cropped_left_upper.nii.gz"))
        nib.save(cropped_img, out_path)

print("Done.")

Done.


Crop right upper quadrant

In [ ]:
import os
import nibabel as nib

input_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/images"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/images_cropped_right_quadrant"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        img = nib.load(path)
        data = img.get_fdata()
        
        # Crop to right upper quadrant cube (RAS orientation)
        # Right: reduce left dimension (axis 0)
        # Upper: reduce inferior dimension (axis 2)
        # Anterior: reduce posterior dimension (axis 1)
        mid_x = data.shape[0] // 2
        mid_y = data.shape[1] // 2
        mid_z = data.shape[2] // 2
        cropped_data = data[mid_x:, mid_y:, :]
        
        cropped_img = nib.Nifti1Image(
            cropped_data,
            img.affine,
            img.header
        )
        
        out_path = os.path.join(output_folder, file.replace(".nii.gz", "_cropped_right_upper.nii.gz"))
        nib.save(cropped_img, out_path)

print("Done.")

Done.


Prepare naming for nnUNet

In [11]:
import os
import shutil

input_folder = '/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/images_cropped_right_quadrant'
output_folder_nnUNet = '/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/input'

# Ensure the output folder for nnUNet exists
os.makedirs(output_folder_nnUNet, exist_ok=True)

# Iterate through the input folder to find NIfTI files and rename them
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.nii.gz'):
            nifti_path = os.path.join(root, file)
            
            # Rename the file with '_0000' before the '.nii.gz'
            new_filename = file.replace('.nii.gz', '_0000.nii.gz')
            new_nifti_path = os.path.join(output_folder_nnUNet, new_filename)
            
            # Copy the file to the new location with the new name
            shutil.copy(nifti_path, new_nifti_path)


Run nnUNet segmentation on the flipped images

In [ ]:
sudo docker run --gpus device=0 --shm-size=10gb --entrypoint=/bin/bash -v /home/jaimebarranco/Desktop/nnUNet:/opt/nnunet_resources -e nnUNet_preprocessed=/opt/nnunet_resources/nnUNet_preprocessed -e nnUNet_raw_data_base=/opt/nnunet_resources/nnUNet_raw_data_base -e RESULTS_FOLDER=/opt/nnunet_resources/nnUNet_trained_models -v /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant:/tmp jaimebarran/fw_gear_aeye:0.0.1 -c "nnUNet_predict -i /tmp/input -o /tmp/output -tr nnUNetTrainerV2 -ctr nnUNetTrainerV2CascadeFullRes -m 3d_fullres -p nnUNetPlansv2.1 -t Task313_Eye"

Flip back the output segmentations

In [ ]:
import os
import nibabel as nib

input_folder = '/home/jaimebarranco/Desktop/tmp/test_inference_flipped_images/flipped_images_nnUNet/output'
output_folder = '/home/jaimebarranco/Desktop/tmp/test_inference_flipped_images/flipped_images_nnUNet/output_flipped_back'

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Function to flip the axis of the image back to the original orientation
def flip_axis_back(img):
    data = img.get_fdata()
    flipped_back_data = data[::-1, :, :]
    flipped_back_img = nib.Nifti1Image(flipped_back_data, img.affine, img.header)
    return flipped_back_img

# Iterate through the input folder to find NIfTI files
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.nii.gz'):
            nifti_path = os.path.join(root, file)
            
            # Load the NIfTI image
            img = nib.load(nifti_path)
            
            # Flip the axis back
            flipped_back_img = flip_axis_back(img)
            
            # Create the same folder structure in the output folder
            relative_path = os.path.relpath(root, input_folder)
            subject_folder = os.path.join(output_folder, relative_path)
            os.makedirs(subject_folder, exist_ok=True)
            
            # Save the flipped back image
            flipped_back_nifti_path = os.path.join(subject_folder, os.path.basename(nifti_path).replace('.nii.gz', '_flipped_back.nii.gz'))
            nib.save(flipped_back_img, flipped_back_nifti_path)

Inverse crop the left eye segmentations to the original image size

In [ ]:
import os
import nibabel as nib
import numpy as np

input_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/output"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/output_inverse_cropped"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        # Load the cropped segmentation
        img = nib.load(path)
        
        # Use dataobj to get original data type or convert to int
        cropped_data = np.asarray(img.dataobj).astype(np.uint8)
        
        print(f"Processing: {file}")
        print(f"Unique labels: {np.unique(cropped_data)}")
        
        # Create empty array with original shape [176, 256, 176]
        original_shape = (176, 256, 176)
        full_data = np.zeros(original_shape, dtype=np.uint8)  # Match the dtype
        
        # Calculate the crop boundaries (same as the forward crop)
        mid_x = original_shape[0] // 2  # 88
        mid_y = original_shape[1] // 2  # 128
        
        # Place the cropped data back: data[:mid_x, mid_y:, :]
        full_data[:mid_x, mid_y:, :] = cropped_data
        
        # Create new header with correct data type
        new_header = img.header.copy()
        new_header.set_data_dtype(np.uint8)
        
        # Create the inverse-cropped NIfTI image
        inverse_cropped_img = nib.Nifti1Image(
            full_data,
            img.affine,
            new_header
        )
        
        out_path = os.path.join(output_folder, file.replace(".nii.gz", "_inverse_cropped.nii.gz"))
        nib.save(inverse_cropped_img, out_path)
        print(f"Saved: {out_path}\n")

print("Done.")

Processing: 0000814968_cropped_left_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/output_inverse_cropped/0000814968_cropped_left_upper_inverse_cropped.nii.gz

Processing: 0000815255_cropped_left_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/output_inverse_cropped/0000815255_cropped_left_upper_inverse_cropped.nii.gz

Processing: 2022160101217_cropped_left_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/output_inverse_cropped/2022160101217_cropped_left_upper_inverse_cropped.nii.gz

Processing: 2022160102225_cropped_left_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_ma

Inverse crop the right eye segmentations to the original image size

In [16]:
import os
import nibabel as nib
import numpy as np

input_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/output"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/output_inverse_cropped"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        # Load the cropped segmentation
        img = nib.load(path)
        
        # Use dataobj to get original data type or convert to int
        cropped_data = np.asarray(img.dataobj).astype(np.uint8)
        
        print(f"Processing: {file}")
        print(f"Unique labels: {np.unique(cropped_data)}")
        
        # Create empty array with original shape [176, 256, 176]
        original_shape = (176, 256, 176)
        full_data = np.zeros(original_shape, dtype=np.uint8)  # Match the dtype
        
        # Calculate the crop boundaries (same as the forward crop)
        mid_x = original_shape[0] // 2  # 88
        mid_y = original_shape[1] // 2  # 128
        
        # Place the cropped data back: data[:mid_x, mid_y:, :]
        full_data[mid_x:, mid_y:, :] = cropped_data
        
        # Create new header with correct data type
        new_header = img.header.copy()
        new_header.set_data_dtype(np.uint8)
        
        # Create the inverse-cropped NIfTI image
        inverse_cropped_img = nib.Nifti1Image(
            full_data,
            img.affine,
            new_header
        )
        
        out_path = os.path.join(output_folder, file.replace(".nii.gz", "_inverse_cropped.nii.gz"))
        nib.save(inverse_cropped_img, out_path)
        print(f"Saved: {out_path}\n")

print("Done.")

Processing: 0000815179_cropped_right_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/output_inverse_cropped/0000815179_cropped_right_upper_inverse_cropped.nii.gz

Processing: 2022160100011_cropped_right_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/output_inverse_cropped/2022160100011_cropped_right_upper_inverse_cropped.nii.gz

Processing: 0000814990_cropped_right_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/output_inverse_cropped/0000814990_cropped_right_upper_inverse_cropped.nii.gz

Processing: 0000814983_cropped_right_upper.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset

Merge segmentations (left-right)

In [17]:
import os
import nibabel as nib
import numpy as np

left_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_left_quadrant/output_inverse_cropped"
right_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/temp_inference_cropped_images_right_quadrant/output_inverse_cropped"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/new_manual_annotations/images/merged_segmentations"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(left_folder):
    if file.endswith(".nii.gz"):
        left_path = os.path.join(left_folder, file)
        right_file = file.replace("_left_", "_right_")  # Adjust naming as needed
        right_path = os.path.join(right_folder, right_file)
        
        if os.path.exists(right_path):
            # Load both segmentations
            left_img = nib.load(left_path)
            right_img = nib.load(right_path)
            
            left_data = np.asarray(left_img.dataobj).astype(np.uint8)
            right_data = np.asarray(right_img.dataobj).astype(np.uint8)
            
            # Merge: combine both segmentations (right overwrites where non-zero)
            merged_data = left_data.copy()
            merged_data[right_data > 0] = right_data[right_data > 0]
            
            # Save merged segmentation
            merged_img = nib.Nifti1Image(merged_data, left_img.affine, left_img.header)
            out_path = os.path.join(output_folder, file.replace("_inverse_cropped", "_merged"))
            nib.save(merged_img, out_path)
            print(f"Merged: {file}")

print("Done.")

Merged: 0000815168_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000814983_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160100984_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160100247_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000814976_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000815262_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000815179_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000815147_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000815259_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160100945_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160102274_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160102457_cropped_left_upper_inverse_cropped.nii.gz
Merged: 0000814990_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160100381_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160102891_cropped_left_upper_inverse_cropped.nii.gz
Merged: 2022160100405_cropped_left_upper_inverse_cropped.nii.gz


# N=1210

Crop left/right upper quadrant

In [3]:
import os
import nibabel as nib

input_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet_left_quadrant"

os.makedirs(output_folder, exist_ok=True)

for file in sorted(os.listdir(input_folder)):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        img = nib.load(path)
        data = img.get_fdata()
        
        # Crop to left upper quadrant cube (RAS orientation)
        # Left: reduce right dimension (axis 0)
        # Upper: reduce inferior dimension (axis 2)
        # Anterior: reduce posterior dimension (axis 1)
        mid_x = data.shape[0] // 2
        mid_y = data.shape[1] // 2
        mid_z = data.shape[2] // 2
        cropped_data = data[:mid_x, mid_y:, :]
        
        cropped_img = nib.Nifti1Image(
            cropped_data,
            img.affine,
            img.header
        )
        
        out_path = os.path.join(output_folder, file)
        nib.save(cropped_img, out_path)

print("Done.")

Done.


In [1]:
import os
import nibabel as nib

input_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet"
output_folder = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet_right_quadrant"

os.makedirs(output_folder, exist_ok=True)

for file in sorted(os.listdir(input_folder)):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        img = nib.load(path)
        data = img.get_fdata()
        
        # Crop to right upper quadrant cube (RAS orientation)
        # Right: reduce left dimension (axis 0)
        # Upper: reduce inferior dimension (axis 2)
        # Anterior: reduce posterior dimension (axis 1)
        mid_x = data.shape[0] // 2
        mid_y = data.shape[1] // 2
        mid_z = data.shape[2] // 2
        cropped_data = data[mid_x:, mid_y:, :]
        
        cropped_img = nib.Nifti1Image(
            cropped_data,
            img.affine,
            img.header
        )
        
        out_path = os.path.join(output_folder, file)
        nib.save(cropped_img, out_path)

print("Done.")

Done.


Run nnUNet

In [ ]:
sudo docker run --gpus device=0 --shm-size=10gb --entrypoint=/bin/bash \
-e nnUNet_preprocessed=/opt/nnunet_resources/nnUNet_preprocessed \
-e nnUNet_raw_data_base=/opt/nnunet_resources/nnUNet_raw_data_base \
-e RESULTS_FOLDER=/opt/nnunet_resources/nnUNet_trained_models \
-v /home/jaimebarranco/Desktop/nnUNet:/opt/nnunet_resources \
-v /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet_left_quadrant:/input \
-v /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_quadrant:/output \
jaimebarran/fw_gear_aeye:0.0.1 -c \
"nnUNet_predict -i /input -o /output -tr nnUNetTrainerV2 -ctr nnUNetTrainerV2CascadeFullRes -m 3d_fullres -p nnUNetPlansv2.1 -t Task313_Eye"

In [ ]:
sudo docker run --gpus device=0 --shm-size=10gb --entrypoint=/bin/bash \
-e nnUNet_preprocessed=/opt/nnunet_resources/nnUNet_preprocessed \
-e nnUNet_raw_data_base=/opt/nnunet_resources/nnUNet_raw_data_base \
-e RESULTS_FOLDER=/opt/nnunet_resources/nnUNet_trained_models \
-v /home/jaimebarranco/Desktop/nnUNet:/opt/nnunet_resources \
-v /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet_right_quadrant:/input \
-v /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_quadrant:/output \
jaimebarran/fw_gear_aeye:0.0.1 -c \
"nnUNet_predict -i /input -o /output -tr nnUNetTrainerV2 -ctr nnUNetTrainerV2CascadeFullRes -m 3d_fullres -p nnUNetPlansv2.1 -t Task313_Eye"

Inverse cropping

In [ ]:
import os
import nibabel as nib
import numpy as np

input_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_quadrant"
output_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        # Load the cropped segmentation
        img = nib.load(path)
        
        # Use dataobj to get original data type or convert to int
        cropped_data = np.asarray(img.dataobj).astype(np.uint8)
        
        print(f"Processing: {file}")
        print(f"Unique labels: {np.unique(cropped_data)}")
        
        # Create empty array with original shape [176, 256, 176]
        original_shape = (176, 256, 176)
        full_data = np.zeros(original_shape, dtype=np.uint8)  # Match the dtype
        
        # Calculate the crop boundaries (same as the forward crop)
        mid_x = original_shape[0] // 2  # 88
        mid_y = original_shape[1] // 2  # 128
        
        # Place the cropped data back: data[:mid_x, mid_y:, :]
        full_data[:mid_x, mid_y:, :] = cropped_data
        
        # Create new header with correct data type
        new_header = img.header.copy()
        new_header.set_data_dtype(np.uint8)
        
        # Create the inverse-cropped NIfTI image
        inverse_cropped_img = nib.Nifti1Image(
            full_data,
            img.affine,
            new_header
        )
        
        out_path = os.path.join(output_folder, file)
        nib.save(inverse_cropped_img, out_path)
        print(f"Saved: {out_path}\n")

print("Done.")

Processing: AEye_2022160102900.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes/AEye_2022160102900.nii.gz

Processing: AEye_2022160103789.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes/AEye_2022160103789.nii.gz

Processing: AEye_2022160103534.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes/AEye_2022160103534.nii.gz

Processing: AEye_2022160103261.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes/AEye_2022160103261.nii.gz

Processing: AEye_2022160101126.n

In [2]:
import os
import nibabel as nib
import numpy as np

input_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_quadrant"
output_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_eyes"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(input_folder):
    if file.endswith(".nii.gz"):
        path = os.path.join(input_folder, file)
        
        # Load the cropped segmentation
        img = nib.load(path)
        
        # Use dataobj to get original data type or convert to int
        cropped_data = np.asarray(img.dataobj).astype(np.uint8)
        
        print(f"Processing: {file}")
        print(f"Unique labels: {np.unique(cropped_data)}")
        
        # Create empty array with original shape [176, 256, 176]
        original_shape = (176, 256, 176)
        full_data = np.zeros(original_shape, dtype=np.uint8)  # Match the dtype
        
        # Calculate the crop boundaries (same as the forward crop)
        mid_x = original_shape[0] // 2  # 88
        mid_y = original_shape[1] // 2  # 128
        
        # Place the cropped data back: data[mid_x:, mid_y:, :]
        full_data[mid_x:, mid_y:, :] = cropped_data
        
        # Create new header with correct data type
        new_header = img.header.copy()
        new_header.set_data_dtype(np.uint8)
        
        # Create the inverse-cropped NIfTI image
        inverse_cropped_img = nib.Nifti1Image(
            full_data,
            img.affine,
            new_header
        )
        
        out_path = os.path.join(output_folder, file)
        nib.save(inverse_cropped_img, out_path)
        print(f"Saved: {out_path}\n")

print("Done.")

Processing: AEye_2022160102900.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_eyes/AEye_2022160102900.nii.gz

Processing: AEye_2022160103789.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_eyes/AEye_2022160103789.nii.gz

Processing: AEye_2022160103534.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_eyes/AEye_2022160103534.nii.gz

Processing: AEye_2022160103261.nii.gz
Unique labels: [0 1 2 3 4 5 6 7 8 9]
Saved: /mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_eyes/AEye_2022160103261.nii.gz

Processing: AEye_20221601011

Merge segmentations (left-right)

In [3]:
import os
import nibabel as nib
import numpy as np

left_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes"
right_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_right_eyes"
output_folder = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_merged_segs"

os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(left_folder):
    if file.endswith(".nii.gz"):
        left_path = os.path.join(left_folder, file)
        right_file = file.replace("_left_", "_right_")  # Adjust naming as needed
        right_path = os.path.join(right_folder, right_file)
        
        if os.path.exists(right_path):
            # Load both segmentations
            left_img = nib.load(left_path)
            right_img = nib.load(right_path)
            
            left_data = np.asarray(left_img.dataobj).astype(np.uint8)
            right_data = np.asarray(right_img.dataobj).astype(np.uint8)
            
            # Merge: combine both segmentations (right overwrites where non-zero)
            merged_data = left_data.copy()
            merged_data[right_data > 0] = right_data[right_data > 0]
            
            # Save merged segmentation
            merged_img = nib.Nifti1Image(merged_data, left_img.affine, left_img.header)
            out_path = os.path.join(output_folder, file.replace("_inverse_cropped", "_merged"))
            nib.save(merged_img, out_path)
            print(f"Merged: {file}")

print("Done.")

Merged: AEye_2022160102900.nii.gz
Merged: AEye_2022160103789.nii.gz
Merged: AEye_2022160103534.nii.gz
Merged: AEye_2022160103261.nii.gz
Merged: AEye_2022160101126.nii.gz
Merged: AEye_2022160100235.nii.gz
Merged: AEye_2022160100342.nii.gz
Merged: AEye_2022160103774.nii.gz
Merged: AEye_2022160101156.nii.gz
Merged: AEye_2022160103380.nii.gz
Merged: AEye_2022160103198.nii.gz
Merged: AEye_2022160103919.nii.gz
Merged: AEye_2022160102825.nii.gz
Merged: AEye_2022160104072.nii.gz
Merged: AEye_2022160101029.nii.gz
Merged: AEye_2022160103688.nii.gz
Merged: AEye_2022160101512.nii.gz
Merged: AEye_2022160103082.nii.gz
Merged: AEye_2022160102887.nii.gz
Merged: AEye_2022160100871.nii.gz
Merged: AEye_2022160102150.nii.gz
Merged: AEye_2022160102714.nii.gz
Merged: AEye_2022160103723.nii.gz
Merged: AEye_2022160103283.nii.gz
Merged: AEye_2022160101299.nii.gz
Merged: AEye_2022160101461.nii.gz
Merged: AEye_2022160100136.nii.gz
Merged: AEye_2022160100014.nii.gz
Merged: AEye_2022160102502.nii.gz
Merged: AEye_2

Select 10 random cases for evaluation

In [ ]:
import os
import nibabel as nib
import numpy as np
import random
import shutil

images_input = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/non_labeled_dataset_nifti_nnunet"
left_folder_input = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_left_eyes"
right_folder_input = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset"
merged_folder_input = "/mnt/sda1/Repos/a-eye/a-eye_segmentation/deep_learning/nnUNet/nnUNet/nnUNet_inference/nnUNet_inference_non_labeled_dataset_merged_segs"

images_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/images"
left_folder_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/left_segs"
right_folder_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/right_segs"
merged_folder_output = "/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/merged_segs"

# Create output directories
os.makedirs(images_output, exist_ok=True)
os.makedirs(left_folder_output, exist_ok=True)
os.makedirs(right_folder_output, exist_ok=True)
os.makedirs(merged_folder_output, exist_ok=True)

# Get all files from images_input
all_files = [f for f in os.listdir(images_input) if f.endswith('.nii.gz')]

# Select 10 random cases
random.seed(10)  # For reproducibility
selected_files = random.sample(all_files, min(10, len(all_files)))

for file in selected_files:
    
    # Copy image
    # Extract digits only (remove 'AEye_' and '_0000')
    # Example: AEye_2022160104419_0000.nii.gz -> 2022160104419.nii.gz
    digits = ''.join(filter(str.isdigit, file.split('_0000')[0]))
    new_name = f"{digits}.nii.gz"
    shutil.copy(os.path.join(images_input, file), os.path.join(images_output, new_name))
    
    # Copy left segmentation
    left_file = f"AEye_{new_name}"
    if os.path.exists(os.path.join(left_folder_input, left_file)):
        shutil.copy(os.path.join(left_folder_input, left_file), os.path.join(left_folder_output, new_name))
    
    # Copy right segmentation
    right_file = f"AEye_{new_name}"
    if os.path.exists(os.path.join(right_folder_input, right_file)):
        shutil.copy(os.path.join(right_folder_input, right_file), os.path.join(right_folder_output, new_name))
    
    # Copy merged segmentation
    merged_file = f"AEye_{new_name}"
    if os.path.exists(os.path.join(merged_folder_input, merged_file)):
        shutil.copy(os.path.join(merged_folder_input, merged_file), os.path.join(merged_folder_output, new_name))

print(f"Copied {len(selected_files)} random cases for evaluation.")

Copied 10 random cases for evaluation.


Create Excel table for evaluation

In [15]:
import pandas as pd
import openpyxl
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation

# Extract subject IDs from selected_files
subject_ids = [''.join(filter(str.isdigit, f.split('_0000')[0])) for f in selected_files]

# Sort subject IDs in ascending order
subject_ids = sorted(subject_ids)

# Create a DataFrame
df = pd.DataFrame({
    'Subject': subject_ids,
    'Score': [''] * len(subject_ids),
    'Comments': [''] * len(subject_ids)
})

# Create Excel file
output_path = '/mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/evaluation_table.xlsx'
df.to_excel(output_path, index=False, sheet_name='Evaluation')

# Add data validation for Score column
wb = openpyxl.load_workbook(output_path)
ws = wb.active

# Create decimal validation (0 to 4)
dv = DataValidation(type="decimal", operator="between", formula1=0, formula2=4, allow_blank=True)
dv.error = 'Please enter a decimal number between 0 and 4'
dv.errorTitle = 'Invalid Entry'
ws.add_data_validation(dv)

# Apply validation to Score column (B2:B11 for 10 rows)
dv.add(f'B2:B{len(subject_ids) + 1}')

wb.save(output_path)
print(f"Excel evaluation table created at: {output_path}")

Excel evaluation table created at: /mnt/sda1/Repos/a-eye/Data/SHIP_dataset/non_labeled_dataset/evaluation_left_eyes/evaluation_table.xlsx
